# Programming Logic Classification (Decision Tree)
Preprocessing, training and evaluation.

## 1. Build the dataset
Snippets are generated from randomised templates; features are extracted by `features.py`.

In [ ]:
import pandas as pd
from generate_dataset import GENERATORS
from features import extract_features, FEATURES

rows = []
for label, gen in GENERATORS.items():
    for _ in range(300):
        c = gen()
        rows.append({'code': c, **extract_features(c), 'label': label})
df = pd.DataFrame(rows).sample(frac=1, random_state=1).reset_index(drop=True)
df.to_csv('dataset.csv', index=False)
df.head()

## 2. Preprocessing and encoding

In [ ]:
print(df.isna().sum().sum(), 'missing values')
print(df.label.value_counts())
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(df.label)
df['label_encoded'] = le.transform(df.label)
df[FEATURES + ['label_encoded']].describe()

## 3. Train the Decision Tree

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
X, y = df[FEATURES], df['label']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
clf = DecisionTreeClassifier(max_depth=8, min_samples_leaf=3, random_state=42).fit(X_tr, y_tr)

## 4. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score
pred = clf.predict(X_te)
print('Accuracy:', accuracy_score(y_te, pred))
print('5-fold CV:', cross_val_score(clf, X, y, cv=5))
print(classification_report(y_te, pred))
ConfusionMatrixDisplay.from_predictions(y_te, pred, xticks_rotation=30)

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
plt.figure(figsize=(18, 8)); plot_tree(clf, feature_names=FEATURES, class_names=list(clf.classes_), filled=True, max_depth=3); plt.show()

## 5. Save the model

In [ ]:
import joblib
joblib.dump({'model': clf, 'features': FEATURES}, 'model.joblib')